# GS-RVFL Tutorial

This notebook provides a comprehensive tutorial on using the Generalized Structured Random Vector Functional Link Network (GS-RVFL).

## Table of Contents
1. Introduction to GS-RVFL
2. Installation and Setup
3. Basic Usage
4. Using Structured Operators
5. Domain-Specific Applications
6. Hyperparameter Tuning
7. Model Evaluation
8. Visualization
9. Advanced Usage


## 1. Introduction to GS-RVFL

GS-RVFL extends the Random Vector Functional Link (RVFL) network by partitioning deterministic inputs into structured direct-link operators and adaptive bias operators. This allows the model to incorporate domain knowledge and preserve the semantic roles of different variables.

### Key Features:
- **Structured Architecture**: Distinguishes between variables based on functional role
- **Closed-Form Learning**: Maintains ridge-regression-based output learning
- **Universal Approximation**: Preserves theoretical guarantees
- **Domain-Specific Operators**: Pre-built operators for motion, sensor, and skeleton data


## 2. Installation and Setup

First, install the required packages:

In [ ]:
# Install dependencies
!pip install numpy scipy scikit-learn pandas matplotlib seaborn

# Install GS-RVFL
!pip install -e ..

## 3. Basic Usage

Let's start with a simple classification example using the Iris dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from gs_rvfl import GSRVFLClassifier
from gs_rvfl.operators import StructuredDirectLink, AdaptiveBias

# Load data
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Number of classes: {len(np.unique(y))}")

In [ ]:
# Create and train GS-RVFL
model = GSRVFLClassifier(
    n_hidden=100,
    lambda_reg=1e-4,
    scale=1.0,
    activation='tanh',
    random_state=42
)

model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

# Evaluate
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Setosa', 'Versicolor', 'Virginica']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

## 4. Using Structured Operators

GS-RVFL allows you to define structured direct-link and adaptive bias operators to incorporate domain knowledge.

In [ ]:
from gs_rvfl.operators import VelocityOperator, DisplacementOperator

# Define operators
g_operators = [VelocityOperator()]  # Structured direct-link
b_operators = [DisplacementOperator()]  # Adaptive bias

# Create model with operators
model_structured = GSRVFLClassifier(
    n_hidden=100,
    lambda_reg=1e-4,
    random_state=42,
    g_operators=g_operators,
    b_operators=b_operators
)

# For this example, we need sequential data
# Create synthetic time series data
X_ts = np.cumsum(np.random.randn(200, 10), axis=0)
y_ts = np.random.randint(0, 2, 200)

X_train_ts, X_test_ts, y_train_ts, y_test_ts = train_test_split(
    X_ts, y_ts, test_size=0.2, random_state=42
)

model_structured.fit(X_train_ts, y_train_ts)
y_pred_ts = model_structured.predict(X_test_ts)

print(f"Accuracy with operators: {accuracy_score(y_test_ts, y_pred_ts):.4f}")
print(f"Number of G features: {model_structured.n_g_features}")
print(f"Number of B features: {model_structured.n_b_features}")
print(f"Total features: {model_structured.n_total_features}")

## 5. Domain-Specific Applications

GS-RVFL provides pre-built operators for different domains.

In [ ]:
from gs_rvfl.operators import (
    create_motion_operators,
    create_sensor_operators,
    create_tabular_operators
)

# Motion operators for action recognition
g_ops, b_ops = create_motion_operators(
    joint_indices=[0, 1, 2, 3, 4],  # Specify joints to use
    time_window=1,
    include_acceleration=True
)

print("Motion Operators:")
print(f"  G operators: {len(g_ops)}")
print(f"  B operators: {len(b_ops)}")
print(f"  G operator types: {[type(op).__name__ for op in g_ops]}")
print(f"  B operator types: {[type(op).__name__ for op in b_ops]}")

# Sensor operators for HAR
g_ops_s, b_ops_s = create_sensor_operators(
    time_window=1,
    include_acceleration=False
)

print("\nSensor Operators:")
print(f"  G operators: {len(g_ops_s)}")
print(f"  B operators: {len(b_ops_s)}")
print(f"  G operator types: {[type(op).__name__ for op in g_ops_s]}")
print(f"  B operator types: {[type(op).__name__ for op in b_ops_s]}")

## 6. Hyperparameter Tuning

GS-RVFL has several hyperparameters that can be tuned for optimal performance.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {
    'n_hidden': [50, 100, 200],
    'lambda_reg': [1e-6, 1e-4, 1e-2],
    'scale': [0.5, 1.0, 2.0],
    'activation': ['sigmoid', 'tanh', 'relu']
}

# Create model
model = GSRVFLClassifier(random_state=42)

# Grid search (using a small subset for demonstration)
print("Running grid search...")
grid_search = GridSearchCV(
    model, param_grid, cv=3, scoring='accuracy', n_jobs=-1
)

# Use subset of data for speed
X_subset = X[:100]
y_subset = y[:100]

grid_search.fit(X_subset, y_subset)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best score: {grid_search.best_score_:.4f}")

## 7. Model Evaluation

Evaluate and compare different configurations.

In [ ]:
import time

# Compare different configurations
configs = [
    ('RVFL (standard)', None, None),
    ('RVFL + G operators', [StructuredDirectLink()], None),
    ('RVFL + B operators', None, [AdaptiveBias()]),
    ('GS-RVFL (full)', [StructuredDirectLink()], [AdaptiveBias()])
]

results = []

for name, g_ops, b_ops in configs:
    model = GSRVFLClassifier(
        n_hidden=50,
        random_state=42,
        g_operators=g_ops or [],
        b_operators=b_ops or []
    )
    
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start
    
    start = time.time()
    y_pred = model.predict(X_test)
    test_time = time.time() - start
    
    acc = accuracy_score(y_test, y_pred)
    
    results.append({
        'Config': name,
        'Accuracy': acc,
        'Train Time (s)': train_time,
        'Test Time (s)': test_time,
        'n_features': model.n_total_features
    })

# Display results
df_results = pd.DataFrame(results)
df_results = df_results.round(4)
display(df_results)

## 8. Visualization

Visualize model behavior and results.

In [ ]:
# Visualize decision boundary (for 2D data)
from sklearn.datasets import make_moons

# Create 2D dataset
X_2d, y_2d = make_moons(n_samples=200, noise=0.15, random_state=42)

# Train model
model_2d = GSRVFLClassifier(n_hidden=50, random_state=42)
model_2d.fit(X_2d, y_2d)

# Create mesh grid
x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                     np.linspace(y_min, y_max, 100))

Z = model_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot
plt.figure(figsize=(10, 8))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_2d, cmap='coolwarm', edgecolor='k')
plt.title('GS-RVFL Decision Boundary')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.colorbar(scatter)
plt.show()

## 9. Advanced Usage

### 9.1 Custom Operators

Create custom operators for specific applications.

In [ ]:
from gs_rvfl.operators import BaseOperator

class CustomOperator(BaseOperator):
    """Custom operator that computes log transformation."""
    
    def __init__(self, eps=1e-10):
        super().__init__(name="CustomLog")
        self.eps = eps
    
    def fit(self, X, y=None):
        X = np.asarray(X)
        self._n_features_out = X.shape[1]
        self._fitted = True
        return self
    
    def transform(self, X):
        X = np.asarray(X)
        self._check_fitted()
        return np.log(np.abs(X) + self.eps)

# Use custom operator
g_ops = [StructuredDirectLink(), CustomOperator()]
model_custom = GSRVFLClassifier(
    n_hidden=50,
    random_state=42,
    g_operators=g_ops
)

model_custom.fit(X_train, y_train)
y_pred_custom = model_custom.predict(X_test)

print(f"Accuracy with custom operator: {accuracy_score(y_test, y_pred_custom):.4f}")

### 9.2 Model Persistence

Save and load trained models.

In [ ]:
import joblib

# Save model
joblib.dump(model, 'gs_rvfl_model.pkl')

# Load model
model_loaded = joblib.load('gs_rvfl_model.pkl')

# Verify
y_pred_loaded = model_loaded.predict(X_test)
print(f"Original accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Loaded accuracy: {accuracy_score(y_test, y_pred_loaded):.4f}")

# Clean up
import os
if os.path.exists('gs_rvfl_model.pkl'):
    os.remove('gs_rvfl_model.pkl')

## Summary

In this tutorial, we covered:

1. **Introduction to GS-RVFL**: Understanding the architecture and benefits
2. **Basic Usage**: Simple classification with Iris dataset
3. **Structured Operators**: Using direct-link and bias operators
4. **Domain-Specific Applications**: Pre-built operators for different domains
5. **Hyperparameter Tuning**: Grid search for optimal parameters
6. **Model Evaluation**: Comparing different configurations
7. **Visualization**: Decision boundary visualization
8. **Advanced Usage**: Custom operators and model persistence

GS-RVFL provides a flexible framework for incorporating domain knowledge into randomized neural networks while maintaining computational efficiency.